# Student Internship RAG System
This notebook implements a Retrieval-Augmented Generation system specialized in parsing complex PDFs with tables using Docling and ChromaDB.

In [ ]:
# 1. Install Dependencies
!pip install -q docling sentence-transformers faiss-cpu rank-bm25 pandas chromadb langchain_groq

In [ ]:
# 2. Imports and Environment Setup
import os
import uuid
import torch
import pandas as pd
import chromadb
from google.colab import files, userdata
from docling.document_converter import DocumentConverter
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from langchain_groq import ChatGroq

# Initialize Models
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Configure Groq LLM
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
llm = ChatGroq(model="llama-3.1-8b-instant")

print("Environment Ready.")

In [ ]:
# 3. Data Loading and Parsing
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

converter = DocumentConverter()
result = converter.convert(pdf_path)
doc = result.document
print(f"Parsed: {pdf_path}")

In [ ]:
# 4. Granular Chunking (Table-Aware)
def create_refined_chunks(doc):
    refined_chunks = []
    # Process Text Elements
    for item, level in doc.iterate_items():
        raw_type = type(item).__name__
        if raw_type == "TableItem": continue
        text = str(item)
        if len(text.strip()) < 5: continue
        refined_chunks.append({
            "id": str(uuid.uuid4())[:8],
            "text": text,
            "metadata": {"type": raw_type, "page": item.prov[0].page_no if item.prov else 0}
        })
    # Process Table Rows
    for table_idx, table in enumerate(doc.tables):
        page_no = table.prov[0].page_no if table.prov else 0
        try:
            df = table.export_to_dataframe(doc=doc)
            for row_idx, row in df.iterrows():
                row_text = f"Table {table_idx} (Page {page_no}): " + " | ".join([f"{col}: {val}" for col, val in row.items()])
                refined_chunks.append({
                    "id": str(uuid.uuid4())[:8],
                    "text": row_text,
                    "metadata": {"type": "TableRow", "table_id": table_idx, "page": page_no}
                })
        except Exception as e: print(f"Table {table_idx} skip: {e}")
    return refined_chunks

chunks = create_refined_chunks(doc)
print(f"Total Chunks: {len(chunks)}")

In [ ]:
# 5. Vector Store & BM25 Initialization
chroma_client = chromadb.EphemeralClient()
collection = chroma_client.create_collection(name="doc_chunks")

# Prepare Data
texts = [c["text"] for c in chunks]
ids = [f"chunk_{c['id']}" for c in chunks]
metadatas = [{"type": c["metadata"]["type"], "page": c["metadata"]["page"]} for c in chunks]

# Populate Vector Store
collection.add(ids=ids, documents=texts, embeddings=embedding_model.encode(texts).tolist(), metadatas=metadatas)

# Populate BM25
tokenized_corpus = [t.lower().split() for t in texts]
bm25 = BM25Okapi(tokenized_corpus)

print("Indexing Complete.")

In [ ]:
# 6. Retrieval Logic (Hybrid + Rerank)
def hybrid_search(query, top_k=10):
    # Vector Search
    v_results = collection.query(query_embeddings=[embedding_model.encode(query).tolist()], n_results=20)
    v_scores = {cid: 1/(1+dist) for cid, dist in zip(v_results['ids'][0], v_results['distances'][0])}

    # Combined Scoring
    combined = []
    bm25_scores = bm25.get_scores(query.lower().split())
    max_bm25 = max(bm25_scores) if max(bm25_scores) > 0 else 1

    for idx, chunk in enumerate(chunks):
        cid = f"chunk_{chunk['id']}"
        score = (0.7 * v_scores.get(cid, 0)) + (0.3 * (bm25_scores[idx]/max_bm25))
        combined.append({"text": chunk["text"], "score": score, "metadata": metadatas[idx]})

    combined.sort(key=lambda x: x["score"], reverse=True)
    return combined[:top_k]

def get_answer(query):
    retrieved = hybrid_search(query)
    pairs = [[query, r["text"]] for r in retrieved]
    rerank_scores = reranker.predict(pairs)

    for i, r in enumerate(retrieved): r["rerank"] = rerank_scores[i]
    retrieved.sort(key=lambda x: x["rerank"], reverse=True)

    context = "\n\n".join([f"[Source: {r['metadata']['type']}, Page: {r['metadata']['page']}]\n{r['text']}" for r in retrieved[:5]])

    prompt = f"Answer the question using ONLY the context.\n\nContext:\n{context}\n\nQuestion: {query}"
    return llm.invoke(prompt).content

## 7. Interactive Querying
Use the cell below to ask questions about the document.

In [ ]:
user_query = "Which industry/company did Deshmukh Pruthviraj Vijaysinh do his internship in?"
answer = get_answer(user_query)
print(f"Q: {user_query}\nA: {answer}")

In [ ]:
!pip install -q docling
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q rank-bm25
!pip install -q pandas
!pip install -q pillow
!pip install -q pymupdf

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch version: 2.11.0+cpu
CUDA available: False
Running on CPU


In [6]:
from google.colab import files

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print("Uploaded:", pdf_path)

Saving Civil_Annual-report-2024_25.pdf to Civil_Annual-report-2024_25.pdf
Uploaded: Civil_Annual-report-2024_25.pdf


In [78]:
##convert the document into structured formattt
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

result = converter.convert(pdf_path)

doc = result.document

print("PDF parsed successfully!")

[INFO] 2026-08-15 19:20:38,082 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-15 19:20:38,197 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-15 19:20:38,199 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-15 19:20:38,278 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-15 19:20:38,289 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-15 19:20:38,290 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-15 19:20:38,350 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-15 19:20:38,495 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[WARNING] 2026-08-15 19:21:25,459 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-08-15 19:21:31,153 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-08-15 19:21:49,585 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-08-15 19:23:04,127 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-08-15 19:23:22,642 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-08-15 19:24:29,565 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-08-15 19:25:12,052 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-08-15 19:26:08,193 [RapidOCR] main.py:132: The text detection result is empty


PDF parsed successfully!


In [8]:
markdown = doc.export_to_markdown()

print(markdown[:10000])

<!-- image -->

## AISSMS

COLLEGE OFENGINEERING

iPls

<!-- image -->

Approved by AICTE, New Delhi, Recognized by Government of Maharashtra

Afi    n   i     tc (Id.No. PU/PN/Engg./093 (1992) Accredited by NAAC with "A+" Grade | NBA - 7 UG Programmes

## DEPARTMENT OF CIVIL ENGINEERING

## ANNUAL REPORT ACADEMIC YEAR 2024 - 25

## VISION OF DEPARTMENT

Nurture  the  talent  in  civil  engineers  to  work  as  global  leaders  for development of society.

## MISSION STATEMENTS OF DEPARTMENT

-  Provide quality education to develop competent civil engineers.
-  Create the awareness among students for sustainable development.
-  Cultivate  the  leadership  qualities  for  becoming successful entrepreneurs.

## 1. FACULTIES ATTENDED FOLLOWING SEMINARS/ WORKSHOPS/ FDP/STTP/CONFERENCES

-  Dr.  U  R Awari,  Dr.  V  S  Chavhan  and  Dr.  S  S  Bhuinyan  attended  FDP  on  Coastal Engineering organized by DYPCOET, Pimpri from 02.09.2024 to 06.09.2024
-  Mrs  K  N  Kulkarni,  Dr.  S  S  

In [9]:
with open("parsed_document.md","w",encoding="utf-8") as f:
  f.write(markdown)
print("Saved parsed_document.md")

Saved parsed_document.md


In [10]:
##hum fixed chunking nhi karenge like 500 chunks and all instead we will divide by section and logically we will connect tem
## toh hum chunk ke saath metadata rakhenge
elements = []

for item, level in doc.iterate_items():

    element = {
        "element_id": len(elements),
        "level": level,
        "element_type": type(item).__name__,
        "text": str(item)
    }

    elements.append(element)

print("Total elements:", len(elements))

Total elements: 302


In [11]:
def classify_element(item):

    item_type = type(item).__name__.lower()

    if "table" in item_type:
        return "table"

    elif "picture" in item_type or "image" in item_type:
        return "image"

    elif "text" in item_type:
        return "text"

    else:
        return "other"

In [12]:
elements = []

for item, level in doc.iterate_items():

    element = {
        "element_id": len(elements),
        "level": level,
        "element_type": classify_element(item),
        "raw_type": type(item).__name__,
        "text": str(item)
    }

    elements.append(element)

In [13]:
from collections import Counter

print(
    Counter(
        element["element_type"]
        for element in elements
    )
)

Counter({'other': 169, 'image': 70, 'text': 55, 'table': 8})


In [14]:
##parent-Child Realtionsh in which it will tell table 2 belongs to section 2
stack = []
elements_with_parent = []

for item, level in doc.iterate_items():

    current_id = len(elements_with_parent)

    element = {
        "element_id": current_id,
        "level": level,
        "element_type": classify_element(item),
        "raw_type": type(item).__name__,
        "text": str(item),
        "parent_id": None
    }

    # Remove elements that are not parents anymore
    while stack and stack[-1]["level"] >= level:
        stack.pop()

    # Current top becomes parent
    if stack:
        element["parent_id"] = stack[-1]["element_id"]

    elements_with_parent.append(element)

    # Current element can become parent
    stack.append({
        "element_id": current_id,
        "level": level
    })

In [15]:
chunks = {
    "chunk_id": 0,

    "text": "...",

    "metadata": {
        "document_id": "uploaded_document",

        "element_ids": [],

        "parent_id": None,

        "element_types": [],

        "levels": [],

        "page_numbers": [],

        "coordinates": []
    }
}

In [16]:
##Embeddings
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [17]:
##vector store
!pip install -q chromadb sentence-transformers rank-bm25
import chromadb
from chromadb.utils import embedding_functions
print("ChromaDB version:", chromadb.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sou

In [18]:
import chromadb

chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

print("ChromaDB initialized!")


ChromaDB initialized!


In [19]:
collection = chroma_client.get_or_create_collection(
    name="document_chunks"
)

print("Collection created!")

Collection created!


In [20]:
print("Number of chunks:", len(chunks))

print("\nFirst chunk:")
print(chunks)

Number of chunks: 3

First chunk:
{'chunk_id': 0, 'text': '...', 'metadata': {'document_id': 'uploaded_document', 'element_ids': [], 'parent_id': None, 'element_types': [], 'levels': [], 'page_numbers': [], 'coordinates': []}}


In [21]:
##list liske elments ids and all should be simple types i.e string,int,boolean therefore keeping all string
def prepare_metadata(metadata):

    clean_metadata = {}

    for key, value in metadata.items():

        if value is None:
            continue

        elif isinstance(value, list):
            clean_metadata[key] = ", ".join(
                str(x) for x in value
            )

        elif isinstance(value, (str, int, float, bool)):
            clean_metadata[key] = value

        else:
            clean_metadata[key] = str(value)

    return clean_metadata

In [22]:
print(
    prepare_metadata(chunks["metadata"])
)

{'document_id': 'uploaded_document', 'element_ids': '', 'element_types': '', 'levels': '', 'page_numbers': '', 'coordinates': ''}


In [23]:
import uuid

# Define DocElement to be compatible with docling items
class DocElement:
    def __init__(self, raw_item, doc_id):
        self.id = str(uuid.uuid4())[:8]
        self.doc_id = doc_id

        # Map docling item type to our internal types
        raw_type_name = type(raw_item).__name__
        if raw_type_name == "SectionHeaderItem":
            self.type = "Title"
        elif raw_type_name == "TextItem":
            self.type = "NarrativeText"
        elif raw_type_name == "ListItem":
            self.type = "ListItem"
        elif raw_type_name == "TableItem":
            self.type = "Table"
        elif raw_type_name == "PictureItem":
            self.type = "Image"
        else:
            self.type = "UncategorizedText" # Fallback, matching original design

        self.page = raw_item.page_number if hasattr(raw_item, 'page_number') else 0
        self.text = str(raw_item)
        self.section = None
        self.related_ids = []
        self.raw = raw_item # Keep reference to the raw docling item

# 1. Create doc_elements from doc.iterate_items()
doc_elements = []
doc_id = "doc_001"
for item, level in doc.iterate_items():
    doc_elements.append(DocElement(item, doc_id))

# 2. Assign sections
current_section = "Untitled"
for el in doc_elements:
    if el.type == "Title":
        current_section = el.text
    el.section = current_section

# 3. Link related elements
def find_element_before(elements_list, index):
    for j in range(index - 1, -1, -1):
        if elements_list[j].type in ("Title", "NarrativeText", "UncategorizedText", "ListItem"):
            return elements_list[j]
    return None

def link_to_nearest_preceding(elements_list):
    for i, el in enumerate(elements_list):
        if el.type not in ("Table", "Image"):
            continue

        preceding = find_element_before(elements_list, i)
        if preceding is not None:
            if preceding.id not in el.related_ids:
                el.related_ids.append(preceding.id)
            if el.id not in preceding.related_ids:
                preceding.related_ids.append(el.id)
    return elements_list

doc_elements = link_to_nearest_preceding(doc_elements)

# 4. Build chunks from the processed doc_elements
def build_chunks_from_docelements(docelements_list):
    chunks = []
    for el in docelements_list:
        if not el.text or len(el.text.strip()) < 3:
            continue

        chunks.append({
            "id": el.id,
            "type": el.type,
            "text": el.text,
            "section": el.section,
            "page": el.page,
            "related_ids": el.related_ids,
        })
    return chunks

chunks = build_chunks_from_docelements(doc_elements)

##generate embeddings for all chunks
texts = [
    chunk["text"]
    for chunk in chunks
]
print(f"Generated {len(chunks)} chunks.")
print(f"First 3 texts: {texts[:3]}")

Generated 302 chunks.
First 3 texts: ["self_ref='#/pictures/0' parent=RefItem(cref='#/body') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.PICTURE: 'picture'> prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=120.88694944787662, t=771.6950821571506, r=177.65518885684364, b=716.698958652177, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 0))] source=[] comments=[] captions=[] references=[] footnotes=[] image=None annotations=[]", "self_ref='#/texts/0' parent=RefItem(cref='#/body') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.SECTION_HEADER: 'section_header'> prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=220.66666666666669, t=775.5866495768229, r=376.0, b=739.2533162434896, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 6))] source=[] comments=[] orig='AISSMS' text='AISSMS' formatting=None hyperlink=None level=1", "self_ref='#/texts/1' parent=RefItem(cref='#/body') c

In [24]:
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Embedding shape: (302, 384)


In [25]:
ids = [
    f"chunk_{chunk['id']}"
    for chunk in chunks
]
print(f"Generated {len(ids)} IDs.")
print(f"First 3 IDs: {ids[:3]}")

Generated 302 IDs.
First 3 IDs: ['chunk_423ab211', 'chunk_5d0371cc', 'chunk_f89eba7d']


In [26]:
print(ids[:10])

['chunk_423ab211', 'chunk_5d0371cc', 'chunk_f89eba7d', 'chunk_42177714', 'chunk_885b2fca', 'chunk_0e678ccd', 'chunk_9b847e3b', 'chunk_b47115a6', 'chunk_e59dd328', 'chunk_d8ca04ca']


In [27]:
metadatas = [
    prepare_metadata({
        "id": chunk["id"],
        "type": chunk["type"],
        "section": chunk["section"],
        "page": chunk["page"],
        "related_ids": chunk["related_ids"]
    })
    for chunk in chunks
]

In [28]:
print(metadatas[0])


{'id': '423ab211', 'type': 'Image', 'section': 'Untitled', 'page': 0, 'related_ids': ''}


In [29]:
collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print(
    "Total chunks stored:",
    collection.count()
)

Total chunks stored: 302


In [30]:
result = collection.get(
    limit=3
)

print(result)

{'ids': ['chunk_423ab211', 'chunk_5d0371cc', 'chunk_f89eba7d'], 'embeddings': None, 'documents': ["self_ref='#/pictures/0' parent=RefItem(cref='#/body') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.PICTURE: 'picture'> prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=120.88694944787662, t=771.6950821571506, r=177.65518885684364, b=716.698958652177, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 0))] source=[] comments=[] captions=[] references=[] footnotes=[] image=None annotations=[]", "self_ref='#/texts/0' parent=RefItem(cref='#/body') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.SECTION_HEADER: 'section_header'> prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=220.66666666666669, t=775.5866495768229, r=376.0, b=739.2533162434896, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 6))] source=[] comments=[] orig='AISSMS' text='AISSMS' formatting=None hyperlink=None lev

In [31]:
##reusable search function
def semantic_search(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )[0]

    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=top_k
    )

    return results

In [32]:
results = semantic_search(
    "What is the main objective?",
    top_k=5
)

In [33]:
for i in range(len(results["ids"][0])):

    print("\n-------------------------")

    print("Result:", i + 1)

    print("Text:")
    print(results["documents"][0][i])

    print("\nMetadata:")
    print(results["metadatas"][0][i])


-------------------------
Result: 1
Text:
self_ref='#/texts/41' parent=RefItem(cref='#/groups/1') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.LIST_ITEM: 'list_item'> prov=[ProvenanceItem(page_no=3, bbox=BoundingBox(l=90.024, t=145.61598291015628, r=526.3899999999996, b=93.74398291015632, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 255))] source=[] comments=[] orig="\uf0b7 Dr. V N Patil Presented paper in 'International Conference on Gender Equality and Women Empowerment  2025'  on  8.3.2025  organized  by  PVG  Pune  on  'Empowering  V oices: Women in Decision making in India- from Historical Legacy to Future Leadership'" text="\uf0b7 Dr. V N Patil Presented paper in 'International Conference on Gender Equality and Women Empowerment  2025'  on  8.3.2025  organized  by  PVG  Pune  on  'Empowering  V oices: Women in Decision making in India- from Historical Legacy to Future Leadership'" formatting=None hyperlink=None enumera

In [34]:
##keyword search-BM25
from rank_bm25 import BM25Okapi

tokenized_corpus = [
    chunk["text"].lower().split()
    for chunk in chunks
]

bm25 = BM25Okapi(
    tokenized_corpus
)

print("BM25 index created!")

BM25 index created!


In [35]:
def bm25_search(query, top_k=5):

    query_tokens = query.lower().split()

    scores = bm25.get_scores(
        query_tokens
    )

    top_indices = scores.argsort()[
        ::-1
    ][:top_k]

    results = []

    for idx in top_indices:

        results.append({
            "index": int(idx),
            "score": float(scores[idx]),
            "text": chunks[idx]["text"],
            "metadata": metadatas[idx] # Corrected to use the metadatas list
        })

    return results

In [36]:
results = bm25_search(
    "builders association of India",
    top_k=5
)

for result in results:

    print("\n----------------")

    print("Score:", result["score"])

    print("Text:", result["text"])


----------------
Score: 12.977480769931049
Text: self_ref='#/texts/422' parent=RefItem(cref='#/groups/12') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.LIST_ITEM: 'list_item'> prov=[ProvenanceItem(page_no=30, bbox=BoundingBox(l=90.024, t=148.85598291015629, r=526.0279999999996, b=117.7639829101563, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 131))] source=[] comments=[] orig='\uf0b7 Sakshi  karande  and  Yogesh  Waghmare  have  received  best  performance  during internship under builder association of India .' text='\uf0b7 Sakshi  karande  and  Yogesh  Waghmare  have  received  best  performance  during internship under builder association of India .' formatting=None hyperlink=None enumerated=False marker=''

----------------
Score: 7.80512140834579
Text: self_ref='#/texts/359' parent=RefItem(cref='#/groups/12') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.LIST_ITEM: 'list_item'> prov

In [37]:
##Hybrid Search (meaning chromadb and BM25 has different based scores there combine making it hybrid)
import numpy as np


def min_max_normalize(scores):

    scores = np.array(
        scores,
        dtype=float
    )

    if len(scores) == 0:
        return scores

    min_score = scores.min()
    max_score = scores.max()

    if max_score == min_score:
        return np.ones_like(scores)

    return (
        (scores - min_score)
        /
        (max_score - min_score)
    )

In [38]:
def get_chroma_results(query, top_k=20):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )[0]

    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=top_k
    )

    return results

In [39]:
def hybrid_search(
    query,
    top_k=5,
    candidate_k=20,
    vector_weight=0.7,
    keyword_weight=0.3
):

    # -----------------------------
    # 1. Chroma semantic search
    # -----------------------------

    chroma_results = get_chroma_results(
        query,
        candidate_k
    )

    chroma_ids = chroma_results["ids"][0]
    chroma_distances = chroma_results["distances"][0]

    # Convert distance into similarity
    vector_scores = {}

    for chunk_id, distance in zip(
        chroma_ids,
        chroma_distances
    ):

        similarity = 1 / (1 + distance)

        vector_scores[chunk_id] = similarity

    # -----------------------------
    # 2. BM25 search
    # -----------------------------

    query_tokens = query.lower().split()

    bm25_scores = bm25.get_scores(
        query_tokens
    )

    # -----------------------------
    # 3. Normalize BM25 scores
    # -----------------------------

    normalized_bm25 = min_max_normalize(
        bm25_scores
    )

    # -----------------------------
    # 4. Combine scores
    # -----------------------------

    combined_results = []

    for idx, chunk in enumerate(chunks):

        chunk_id = f"chunk_{chunk['id']}"

        semantic_score = vector_scores.get(
            chunk_id,
            0.0
        )

        keyword_score = normalized_bm25[idx]

        final_score = (
            vector_weight * semantic_score
            +
            keyword_weight * keyword_score
        )

        combined_results.append({

            "chunk_id": chunk_id,

            "index": idx,

            "score": float(final_score),

            "text": chunk["text"],

            "metadata": metadatas[idx]

        })

    # -----------------------------
    # 5. Sort
    # -----------------------------

    combined_results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return combined_results[:top_k]

In [40]:
query = input(
    "Exhibition"
)

results = hybrid_search(
    query,
    top_k=5
)

Exhibitionexhibition


In [41]:
##display
for i, result in enumerate(results):

    print("\n" + "=" * 70)

    print(
        "Rank:",
        i + 1
    )

    print(
        "Score:",
        result["score"]
    )

    print(
        "\nText:",
        result["text"]
    )

    print(
        "\nMetadata:",
        result["metadata"]
    )



Rank: 1
Score: 0.5868468821242756

Text: self_ref='#/texts/56' parent=RefItem(cref='#/pictures/2') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.CAPTION: 'caption'> prov=[ProvenanceItem(page_no=5, bbox=BoundingBox(l=112.34, t=465.82406291015633, r=521.86, b=456.5380847283381, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 87))] source=[] comments=[] orig='Poster Competition cum Exhibition for DHS Subject at BE Civil - Judge Prof Snehal Dethe' text='Poster Competition cum Exhibition for DHS Subject at BE Civil - Judge Prof Snehal Dethe' formatting=None hyperlink=None

Metadata: {'id': '9ff858a8', 'type': 'NarrativeText', 'section': "self_ref='#/texts/54' parent=RefItem(cref='#/body') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.SECTION_HEADER: 'section_header'> prov=[ProvenanceItem(page_no=5, bbox=BoundingBox(l=79.104, t=766.8639829101563, r=499.06, b=736.1305283647017, coord_origin=<CoordO

Reranker:which BM25 or Semantic search

In [42]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Reranker loaded!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Reranker loaded!


In [43]:
def rerank_results(
    query,
    results,
    top_k=5
):

    pairs = [
        [query, result["text"]]
        for result in results
    ]

    scores = reranker.predict(
        pairs
    )

    reranked = []

    for result, score in zip(
        results,
        scores
    ):

        result_copy = result.copy()

        result_copy["rerank_score"] = float(
            score
        )

        reranked.append(
            result_copy
        )

    reranked.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return reranked[:top_k]

In [44]:
hybrid_results = hybrid_search(
    query,
    top_k=20
)

final_results = rerank_results(
    query,
    hybrid_results,
    top_k=5
)

In [45]:
for i, result in enumerate(final_results):

    print("\n" + "=" * 70)

    print("Rank:", i + 1)

    print(
        "Rerank score:",
        result["rerank_score"]
    )

    print(
        "\nText:",
        result["text"]
    )

    print(
        "\nMetadata:",
        result["metadata"]
    )


Rank: 1
Rerank score: -2.6856393814086914

Text: self_ref='#/texts/56' parent=RefItem(cref='#/pictures/2') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.CAPTION: 'caption'> prov=[ProvenanceItem(page_no=5, bbox=BoundingBox(l=112.34, t=465.82406291015633, r=521.86, b=456.5380847283381, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 87))] source=[] comments=[] orig='Poster Competition cum Exhibition for DHS Subject at BE Civil - Judge Prof Snehal Dethe' text='Poster Competition cum Exhibition for DHS Subject at BE Civil - Judge Prof Snehal Dethe' formatting=None hyperlink=None

Metadata: {'id': '9ff858a8', 'type': 'NarrativeText', 'section': "self_ref='#/texts/54' parent=RefItem(cref='#/body') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.SECTION_HEADER: 'section_header'> prov=[ProvenanceItem(page_no=5, bbox=BoundingBox(l=79.104, t=766.8639829101563, r=499.06, b=736.1305283647017, coord_origin

In [46]:
##LLm
!pip install langchain_groq
from langchain_groq import ChatGroq
import os
from google.colab import userdata

# Retrieve the API key from Colab secrets
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# Set it as an environment variable (often used by libraries like langchain-openai)
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
llm=ChatGroq(model="llama-3.1-8b-instant")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.5 MB/s eta 0:00:00


In [47]:
# Check what is stored inside ChromaDB

print("Number of documents:", collection.count())

sample = collection.get(limit=3)

print("\nSample documents:")
for i, doc in enumerate(sample["documents"]):
    print(f"\n--- Document {i+1} ---")
    print(doc[:500])

print("\nSample metadata:")
for metadata in sample["metadatas"]:
    print(metadata)

Number of documents: 302

Sample documents:

--- Document 1 ---
self_ref='#/pictures/0' parent=RefItem(cref='#/body') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.PICTURE: 'picture'> prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=120.88694944787662, t=771.6950821571506, r=177.65518885684364, b=716.698958652177, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 0))] source=[] comments=[] captions=[] references=[] footnotes=[] image=None annotations=[]

--- Document 2 ---
self_ref='#/texts/0' parent=RefItem(cref='#/body') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.SECTION_HEADER: 'section_header'> prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=220.66666666666669, t=775.5866495768229, r=376.0, b=739.2533162434896, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 6))] source=[] comments=[] orig='AISSMS' text='AISSMS' formatting=None hyperlink=None level=1

--- Document

In [48]:
print(final_results)

[{'chunk_id': 'chunk_9ff858a8', 'index': 76, 'score': 0.5868468821242756, 'text': "self_ref='#/texts/56' parent=RefItem(cref='#/pictures/2') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.CAPTION: 'caption'> prov=[ProvenanceItem(page_no=5, bbox=BoundingBox(l=112.34, t=465.82406291015633, r=521.86, b=456.5380847283381, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 87))] source=[] comments=[] orig='Poster Competition cum Exhibition for DHS Subject at BE Civil - Judge Prof Snehal Dethe' text='Poster Competition cum Exhibition for DHS Subject at BE Civil - Judge Prof Snehal Dethe' formatting=None hyperlink=None", 'metadata': {'id': '9ff858a8', 'type': 'NarrativeText', 'section': "self_ref='#/texts/54' parent=RefItem(cref='#/body') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.SECTION_HEADER: 'section_header'> prov=[ProvenanceItem(page_no=5, bbox=BoundingBox(l=79.104, t=766.8639829101563, r=499.0

In [49]:
print(rerank_score)

NameError: name 'rerank_score' is not defined

Context Expansion


In [50]:
def expand_context(
    retrieved_chunks,
    chunk_map,
    window_size=2
):
    """
    Expands retrieved chunks using neighboring chunks.

    The expansion is structure-aware:
    - Uses chunk index to find nearby chunks
    - Prefers chunks from the same section
    - Keeps the original retrieved chunk
    """

    expanded_results = []

    for chunk in retrieved_chunks:

        metadata = chunk["metadata"]

        current_index = chunk.get("index")

        if current_index is None:
            current_index = metadata.get("index")

        if current_index is None:
            # If index is unavailable, keep original chunk
            expanded_results.append(chunk)
            continue

        current_section = metadata.get("section")

        context_chunks = []

        # Look before and after the retrieved chunk
        for offset in range(
            -window_size,
            window_size + 1
        ):

            neighbor_index = current_index + offset

            if neighbor_index not in chunk_map:
                continue

            neighbor = chunk_map[neighbor_index]

            neighbor_metadata = neighbor["metadata"]

            neighbor_section = neighbor_metadata.get("section")

            # Keep chunks belonging to the same section
            if neighbor_section == current_section:

                context_chunks.append({
                    "index": neighbor_index,
                    "text": neighbor["text"],
                    "metadata": neighbor_metadata,
                    "source": (
                        "retrieved"
                        if offset == 0
                        else "expanded"
                    )
                })

        # Sort according to original document order
        context_chunks.sort(
            key=lambda x: x["index"]
        )

        expanded_results.append({
            "retrieved_index": current_index,
            "chunks": context_chunks
        })

    return expanded_results

In [51]:
chunk_map = {
    idx: {
        "text": chunks[idx]["text"],
        "metadata": metadatas[idx]
    }
    for idx in range(len(chunks))
}

expanded_results = expand_context(
    retrieved_chunks=final_results,
    chunk_map=chunk_map,
    window_size=2
)

In [52]:
for result in expanded_results:

    print("\n" + "=" * 80)

    print(
        "Retrieved chunk index:",
        result["retrieved_index"]
    )

    print(
        "Number of context chunks:",
        len(result["chunks"])
    )

    for chunk in result["chunks"]:

        print(
            f"\n[{chunk['source']}] "
            f"Index: {chunk['index']}"
        )

        print(
            chunk["text"][:300]
        )


Retrieved chunk index: 76
Number of context chunks: 5

[expanded] Index: 74
self_ref='#/texts/81' parent=RefItem(cref='#/groups/2') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.LIST_ITEM: 'list_item'> prov=[ProvenanceItem(page_no=6, bbox=BoundingBox(l=90.024, t=302.4859829101563, r=526.318, b=271.3639829101562, coord_origin=<CoordOrigi

[expanded] Index: 75
self_ref='#/pictures/2' parent=RefItem(cref='#/body') children=[RefItem(cref='#/texts/56'), RefItem(cref='#/texts/57'), RefItem(cref='#/texts/58'), RefItem(cref='#/texts/59'), RefItem(cref='#/texts/60'), RefItem(cref='#/texts/61'), RefItem(cref='#/texts/62'), RefItem(cref='#/texts/63')] content_laye

[retrieved] Index: 76
self_ref='#/texts/56' parent=RefItem(cref='#/pictures/2') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.CAPTION: 'caption'> prov=[ProvenanceItem(page_no=5, bbox=BoundingBox(l=112.34, t=465.82406291015633, r=521.86, b=456.5380847283381, co

In [53]:
##making realtionship of chunk with table ,image and captoon and all
related_map = {}

for document_text, metadata_dict in zip(
    texts, # Changed from all_data["documents"]
    metadatas # Changed from all_data["metadatas"]
):

    chunk_id = metadata_dict.get("id")

    if chunk_id:
        related_map[chunk_id] = {
            "text": document_text,
            "metadata": metadata_dict
        }

print("Related content entries:", len(related_map))

Related content entries: 302


In [54]:
def get_related_chunks(metadata, related_map):
    """
    Find chunks related to the current chunk
    using the related_ids metadata.
    """

    related_chunks = []

    related_ids = metadata.get("related_ids", "")

    if not related_ids:
        return related_chunks

    # In case multiple IDs are stored
    ids = [
        x.strip()
        for x in related_ids.split(",")
        if x.strip()
    ]

    for related_id in ids:

        if related_id in related_map:

            related = related_map[related_id]

            related_chunks.append({
                "text": related["text"],
                "metadata": related["metadata"],
                "source": "related"
            })

    return related_chunks

In [55]:
test_chunk = final_results[0]

print("Chunk:")
print(test_chunk["text"][:500])

print("\nMetadata:")
print(test_chunk["metadata"])

Chunk:
self_ref='#/texts/56' parent=RefItem(cref='#/pictures/2') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.CAPTION: 'caption'> prov=[ProvenanceItem(page_no=5, bbox=BoundingBox(l=112.34, t=465.82406291015633, r=521.86, b=456.5380847283381, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 87))] source=[] comments=[] orig='Poster Competition cum Exhibition for DHS Subject at BE Civil - Judge Prof Snehal Dethe' text='Poster Competition cum Exhibit

Metadata:
{'id': '9ff858a8', 'type': 'NarrativeText', 'section': "self_ref='#/texts/54' parent=RefItem(cref='#/body') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.SECTION_HEADER: 'section_header'> prov=[ProvenanceItem(page_no=5, bbox=BoundingBox(l=79.104, t=766.8639829101563, r=499.06, b=736.1305283647017, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 80))] source=[] comments=[] orig='2. FACULTIES ORGANISED SEMINARS/ EXPERT LEC

In [56]:
related = get_related_chunks(
    test_chunk["metadata"],
    related_map
)

print("\nRelated chunks:", len(related))

for item in related:

    print("\n--- Related ---")
    print(item["metadata"])
    print(item["text"][:300])


Related chunks: 1

--- Related ---
{'id': '9e6021ce', 'type': 'Image', 'section': "self_ref='#/texts/54' parent=RefItem(cref='#/body') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.SECTION_HEADER: 'section_header'> prov=[ProvenanceItem(page_no=5, bbox=BoundingBox(l=79.104, t=766.8639829101563, r=499.06, b=736.1305283647017, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 80))] source=[] comments=[] orig='2. FACULTIES ORGANISED SEMINARS/ EXPERT LECTURE /WORKSHOPS/ FDP/STTP/CONFERENCES' text='2. FACULTIES ORGANISED SEMINARS/ EXPERT LECTURE /WORKSHOPS/ FDP/STTP/CONFERENCES' formatting=None hyperlink=None level=1", 'page': 0, 'related_ids': '9ff858a8'}
self_ref='#/pictures/3' parent=RefItem(cref='#/body') children=[RefItem(cref='#/texts/82')] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.PICTURE: 'picture'> prov=[ProvenanceItem(page_no=6, bbox=BoundingBox(l=161.2601301567848, t=264.7188593552596, r=424.2148

In [57]:
def structure_aware_context_expansion(
    retrieved_chunks,
    chunk_map,
    related_map,
    window_size=2
):
    """
    Structure-aware context expansion.

    Expands each retrieved chunk using:

    1. Neighboring chunks
    2. Same-section chunks
    3. Related content
    """

    expanded_results = []

    for retrieved in retrieved_chunks:

        current_index = retrieved.get("index")

        if current_index is None:
            current_index = retrieved["metadata"].get("index")

        current_metadata = retrieved["metadata"]

        current_section = current_metadata.get("section")

        context = []

        # ------------------------------------------------
        # 1. Add neighboring chunks
        # ------------------------------------------------

        for offset in range(
            -window_size,
            window_size + 1
        ):

            neighbor_index = current_index + offset

            if neighbor_index not in chunk_map:
                continue

            neighbor = chunk_map[neighbor_index]

            neighbor_metadata = neighbor["metadata"]

            # Only use chunks from same section
            if (
                neighbor_metadata.get("section")
                == current_section
            ):

                context.append({
                    "index": neighbor_index,
                    "text": neighbor["text"],
                    "metadata": neighbor_metadata,
                    "source": (
                        "retrieved"
                        if offset == 0
                        else "neighbor"
                    )
                })

        # ------------------------------------------------
        # 2. Add related content
        # ------------------------------------------------

        related_chunks = get_related_chunks(
            current_metadata,
            related_map
        )

        for related in related_chunks:

            context.append({
                "index": None,
                "text": related["text"],
                "metadata": related["metadata"],
                "source": "related"
            })

        # ------------------------------------------------
        # 3. Remove duplicates
        # ------------------------------------------------

        unique_context = []

        seen = set()

        for item in context:

            key = (
                item["text"],
                item["metadata"].get("id")
            )

            if key not in seen:

                seen.add(key)

                unique_context.append(item)

        # ------------------------------------------------
        # 4. Sort neighboring chunks
        # ------------------------------------------------

        indexed = [
            x for x in unique_context
            if x["index"] is not None
        ]

        non_indexed = [
            x for x in unique_context
            if x["index"] is None
        ]

        indexed.sort(
            key=lambda x: x["index"]
        )

        unique_context = indexed + non_indexed

        # ------------------------------------------------
        # 5. Save result
        # ------------------------------------------------

        expanded_results.append({

            "retrieved_chunk": retrieved,

            "context": unique_context
        })

    return expanded_results

In [58]:
expanded_results = structure_aware_context_expansion(
    retrieved_chunks=final_results,
    chunk_map=chunk_map,
    related_map=related_map,
    window_size=2
)

In [59]:
for result in expanded_results:

    print("\n")
    print("=" * 100)

    retrieved = result["retrieved_chunk"]

    print(
        "RETRIEVED CHUNK:",
        retrieved["chunk_id"]
    )

    print(
        "RERANK SCORE:",
        retrieved.get("rerank_score")
    )

    print("\nEXPANDED CONTEXT:")

    for item in result["context"]:

        print("\n-----------------------------")

        print(
            "Source:",
            item["source"]
        )

        print(
            "Type:",
            item["metadata"].get("type")
        )

        print(
            "Index:",
            item["index"]
        )

        print(
            "Text:",
            item["text"][:500]
        )



RETRIEVED CHUNK: chunk_9ff858a8
RERANK SCORE: -2.6856393814086914

EXPANDED CONTEXT:

-----------------------------
Source: neighbor
Type: ListItem
Index: 74
Text: self_ref='#/texts/81' parent=RefItem(cref='#/groups/2') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.LIST_ITEM: 'list_item'> prov=[ProvenanceItem(page_no=6, bbox=BoundingBox(l=90.024, t=302.4859829101563, r=526.318, b=271.3639829101562, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 171))] source=[] comments=[] orig="\uf0b7 Dr.  S.  S.  Bhuinyan  has  organised  Expert  Lecture  Report  on  'Bridging  Theory  and Practice: Industry Deman

-----------------------------
Source: neighbor
Type: Image
Index: 75
Text: self_ref='#/pictures/2' parent=RefItem(cref='#/body') children=[RefItem(cref='#/texts/56'), RefItem(cref='#/texts/57'), RefItem(cref='#/texts/58'), RefItem(cref='#/texts/59'), RefItem(cref='#/texts/60'), RefItem(cref='#/texts/61'), RefItem(cref='#/texts/62'

Context Assembly


In [60]:
##groq will not understand the json format given by context expansion there structurized
def assemble_context(expanded_results):
    """
    Convert expanded retrieval results into
    a clean context string for the LLM.
    """

    context_parts = []

    for result_num, result in enumerate(expanded_results, start=1):

        context_parts.append(
            f"\n===== RESULT {result_num} ====="
        )

        for item in result["context"]:

            metadata = item["metadata"]

            source_type = metadata.get(
                "type",
                "Unknown"
            )

            section = metadata.get(
                "section",
                "Unknown"
            )

            page = metadata.get(
                "page",
                "Unknown"
            )

            source = item.get(
                "source",
                "unknown"
            )

            text = item["text"]

            context_parts.append(
                f"""
[Source: {source}]
[Type: {source_type}]
[Section: {section}]
[Page: {page}]

{text}
"""
            )

    return "\n".join(context_parts)

In [61]:
context = assemble_context(
    expanded_results
)

print(context[:5000])


===== RESULT 1 =====

[Source: neighbor]
[Type: ListItem]
[Section: self_ref='#/texts/54' parent=RefItem(cref='#/body') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.SECTION_HEADER: 'section_header'> prov=[ProvenanceItem(page_no=5, bbox=BoundingBox(l=79.104, t=766.8639829101563, r=499.06, b=736.1305283647017, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 80))] source=[] comments=[] orig='2. FACULTIES ORGANISED SEMINARS/ EXPERT LECTURE /WORKSHOPS/ FDP/STTP/CONFERENCES' text='2. FACULTIES ORGANISED SEMINARS/ EXPERT LECTURE /WORKSHOPS/ FDP/STTP/CONFERENCES' formatting=None hyperlink=None level=1]
[Page: 0]

self_ref='#/texts/81' parent=RefItem(cref='#/groups/2') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.LIST_ITEM: 'list_item'> prov=[ProvenanceItem(page_no=6, bbox=BoundingBox(l=90.024, t=302.4859829101563, r=526.318, b=271.3639829101562, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>)

In [62]:
def extract_sources(expanded_results):

    sources = []

    for result in expanded_results:

        for item in result["context"]:

            metadata = item["metadata"]

            source = {
                "page": metadata.get("page"),
                "type": metadata.get("type"),
                "section": metadata.get("section"),
                "chunk_id": metadata.get("id")
            }

            sources.append(source)

    return sources

In [63]:
sources = extract_sources(
    expanded_results
)

print(sources[:5])

[{'page': 0, 'type': 'ListItem', 'section': "self_ref='#/texts/54' parent=RefItem(cref='#/body') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.SECTION_HEADER: 'section_header'> prov=[ProvenanceItem(page_no=5, bbox=BoundingBox(l=79.104, t=766.8639829101563, r=499.06, b=736.1305283647017, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 80))] source=[] comments=[] orig='2. FACULTIES ORGANISED SEMINARS/ EXPERT LECTURE /WORKSHOPS/ FDP/STTP/CONFERENCES' text='2. FACULTIES ORGANISED SEMINARS/ EXPERT LECTURE /WORKSHOPS/ FDP/STTP/CONFERENCES' formatting=None hyperlink=None level=1", 'chunk_id': '677bfb92'}, {'page': 0, 'type': 'Image', 'section': "self_ref='#/texts/54' parent=RefItem(cref='#/body') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.SECTION_HEADER: 'section_header'> prov=[ProvenanceItem(page_no=5, bbox=BoundingBox(l=79.104, t=766.8639829101563, r=499.06, b=736.1305283647017, coord_origin=<C

In [64]:
def create_rag_prompt(query, context):

    prompt = f"""
You are a document question-answering assistant.

Answer the user's question using ONLY the information
provided in the context below.

If the answer cannot be found in the context,
clearly say that the information is not available
in the provided document.

Do not invent facts.

USER QUESTION:
{query}

DOCUMENT CONTEXT:
{context}

INSTRUCTIONS:
1. Answer the question directly.
2. Use the provided document context.
3. Do not hallucinate information.
4. If multiple pieces of context are relevant, combine them.
5. Keep the answer clear and concise.
"""

    return prompt

In [65]:
##LLm
!pip install langchain_groq
from langchain_groq import ChatGroq
import os
from google.colab import userdata

# Retrieve the API key from Colab secrets
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# Set it as an environment variable (often used by libraries like langchain-openai)
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
llm=ChatGroq(model="llama-3.1-8b-instant")

In [66]:
def generate_answer(query, context, max_context_chars=10000):
    # Truncate context to fit within typical LLM token limits
    if len(context) > max_context_chars:
        context = context[:max_context_chars] + "\n[... Context truncated due to length limits ...]\n"

    prompt = create_rag_prompt(
        query,
        context
    )

    response = llm.invoke(
        prompt
    )

    return response.text

In [70]:
query = "the student name with Deshmukh Pruthviraj Vijaysinh is of which industry "

In [71]:
answer = generate_answer(
    query,
    context
)

print(answer)

The information is not available in the provided document. The context does not mention the student name "Deshmukh Pruthviraj Vijaysinh" or their industry.


In [72]:
def ask_document(
    query,
    collection,
    chunk_map,
    related_map,
    reranked_results
):

    # -----------------------------------------
    # STEP 1: Context Expansion
    # -----------------------------------------

    expanded_results = structure_aware_context_expansion(
        retrieved_chunks=reranked_results,
        chunk_map=chunk_map,
        related_map=related_map,
        window_size=2
    )

    # -----------------------------------------
    # STEP 2: Context Assembly
    # -----------------------------------------

    context = assemble_context(
        expanded_results
    )

    # -----------------------------------------
    # STEP 3: Generate Answer
    # -----------------------------------------

    answer = generate_answer(
        query,
        context
    )

    # -----------------------------------------
    # STEP 4: Sources
    # -----------------------------------------

    sources = extract_sources(
        expanded_results
    )

    return {
        "query": query,
        "answer": answer,
        "context": context,
        "sources": sources
    }

In [75]:
# Search all ChromaDB documents for the student's name

all_data = collection.get(
    include=["documents", "metadatas"]
)

matches = []

for document, metadata in zip(
    all_data["documents"],
    all_data["metadatas"]
):

    if "Deshmukh" in document or "Pruthviraj" in document:
        matches.append({
            "text": document,
            "metadata": metadata
        })

print("Matches found:", len(matches))

for item in matches:
    print("\n" + "=" * 80)
    print(item["text"])
    print("\nMetadata:")
    print(item["metadata"])

Matches found: 3

self_ref='#/texts/73' parent=RefItem(cref='#/groups/2') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.LIST_ITEM: 'list_item'> prov=[ProvenanceItem(page_no=6, bbox=BoundingBox(l=90.024, t=702.6159829101563, r=526.3839999999999, b=671.6439829101563, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 140))] source=[] comments=[] orig='\uf0b7 Dr. S. R. Parekar and Dr. P.R. Satarkar has arranged expert session on Truss Analysis using ANSYS, 10/10/2024, Expert - Mr. P.V. Deshmukh.' text='\uf0b7 Dr. S. R. Parekar and Dr. P.R. Satarkar has arranged expert session on Truss Analysis using ANSYS, 10/10/2024, Expert - Mr. P.V. Deshmukh.' formatting=None hyperlink=None enumerated=False marker=''

Metadata:
{'page': 0, 'type': 'ListItem', 'related_ids': '', 'section': "self_ref='#/texts/54' parent=RefItem(cref='#/body') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.SECTION_HEADER: 'section_h

In [76]:
query = "Deshmukh Pruthviraj Vijaysinh internship company"

# Use your existing embedding function
query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

for i in range(len(results["documents"][0])):

    print("\n" + "=" * 100)

    print("RESULT:", i + 1)

    print("DISTANCE:",
          results["distances"][0][i])

    print("\nTEXT:")
    print(
        results["documents"][0][i][:2000]
    )

    print("\nMETADATA:")
    print(
        results["metadatas"][0][i]
    )


RESULT: 1
DISTANCE: 1.401206135749817

TEXT:
self_ref='#/texts/408' parent=RefItem(cref='#/groups/12') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.LIST_ITEM: 'list_item'> prov=[ProvenanceItem(page_no=30, bbox=BoundingBox(l=90.024, t=491.38598291015626, r=526.3059999999998, b=438.1899829101563, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 190))] source=[] comments=[] orig='\uf0b7 Maitreyee  WADATKAR got  selected  for  inter  district  youth  exchange  programme from 10 th Feb  to  14 th Feb  2025  under  Nehru  yuva  kendra  -    Mera  yuva  Bharat  ,  at Wardha .' text='\uf0b7 Maitreyee  WADATKAR got  selected  for  inter  district  youth  exchange  programme from 10 th Feb  to  14 th Feb  2025  under  Nehru  yuva  kendra  -    Mera  yuva  Bharat  ,  at Wardha .' formatting=None hyperlink=None enumerated=False marker=''

METADATA:
{'id': 'c90570b7', 'page': 0, 'related_ids': '', 'section': 'self_ref=\'#/texts/323\' parent=

In [ ]:
%whos

In [79]:
print("Number of tables:", len(doc.tables))

for i, table in enumerate(doc.tables):
    print("\n" + "=" * 80)
    print("TABLE:", i)
    print("Page:", table.prov[0].page_no if table.prov else "Unknown")

    try:
        print(table.export_to_dataframe(doc=doc).head())
    except Exception as e:
        print("Could not convert to dataframe:", e)

Number of tables: 8

TABLE: 0
Page: 20
  Sr. No. Faculty Coordinator  \
0       1      Dr P B Nangare   
1       2      Dr S R Parekar   
2       3       Dr. V N Patil   
3       4    Dr. R D Nalawade   
4       5      Dr. D V Wadkar   

                                Name of Organisation     Date of MoU Signed  \
0                      Builders Association of India                   2021   
1                                        Ferrocement                   2021   
2  1. Struengineers (India) Pvt. Ltd, Pune 2. Neh...  18/05/2021 07/11/2022   
3                           Shree Samarth Associates             20/05/2022   
4  1. Avishkar Realty,Pune. Mr. Kavish Takawani 2...  21/07/2021 01/09/2022   

   Valid upto 00/00/0000  
0               Lifetime  
1               Lifetime  
2  18/05/2026 07/11/2027  
3             19/05/2027  
4  21/07/2026 01/09/2027  

TABLE: 1
Page: 21
    0                 1                                                  2  \
0  13   Dr S S Bhuniyan  (GE

In [80]:
def create_table_row_chunks(doc):

    table_chunks = []

    for table_id, table in enumerate(doc.tables):

        try:
            df = table.export_to_dataframe(doc=doc)

        except Exception as e:
            print(
                f"Skipping table {table_id}: {e}"
            )
            continue

        # Clean column names
        columns = [
            str(col).strip()
            for col in df.columns
        ]

        # Get page number from Docling
        page_no = (
            table.prov[0].page_no
            if table.prov
            else None
        )

        # Process every row
        for row_index, row in df.iterrows():

            row_parts = []

            for column, value in zip(
                columns,
                row.values
            ):

                value = str(value).strip()

                # Skip empty cells
                if value.lower() in {
                    "",
                    "nan",
                    "none"
                }:
                    continue

                row_parts.append(
                    f"{column}: {value}"
                )

            if not row_parts:
                continue

            # Generic semantic representation
            text = (
                f"TABLE {table_id}\n"
                + "\n".join(row_parts)
            )

            metadata = {

                "element_type": "table_row",

                "table_id": str(table_id),

                "row_index": int(row_index),

                "page": page_no,

                "num_columns": len(columns)
            }

            table_chunks.append({

                "text": text,

                "metadata": metadata
            })

    return table_chunks

In [83]:
def create_refined_chunks(doc, doc_id="doc_001"):
    refined_chunks = []

    # 1. Process regular items (Text, Titles, etc.)
    for item, level in doc.iterate_items():
        raw_type = type(item).__name__
        # Skip tables here, we handle them separately for better granularity
        if raw_type == "TableItem":
            continue

        text = str(item)
        if len(text.strip()) < 5: continue

        refined_chunks.append({
            "id": str(uuid.uuid4())[:8],
            "text": text,
            "metadata": {
                "type": raw_type,
                "page": item.prov[0].page_no if item.prov else 0,
                "section": "General",
                "element_type": "text"
            }
        })

    # 2. Process Tables row-by-row
    for table_idx, table in enumerate(doc.tables):
        page_no = table.prov[0].page_no if table.prov else 0
        try:
            df = table.export_to_dataframe(doc=doc)
            # Convert each row to a descriptive string
            for row_idx, row in df.iterrows():
                row_text = f"Table {table_idx} (Page {page_no}): " + " | ".join([f"{col}: {val}" for col, val in row.items()])

                refined_chunks.append({
                    "id": str(uuid.uuid4())[:8],
                    "text": row_text,
                    "metadata": {
                        "type": "TableRow",
                        "table_id": table_idx,
                        "page": page_no,
                        "element_type": "table_row"
                    }
                })
        except Exception as e:
            print(f"Error processing table {table_idx}: {e}")

    return refined_chunks

# Re-generate chunks with table row support
new_chunks = create_refined_chunks(doc)
print(f"Created {len(new_chunks)} refined chunks (including table rows).")

Created 406 refined chunks (including table rows).


In [84]:
# Update the Vector Store with the new granular chunks
collection.delete(ids=collection.get()["ids"]) # Clear old chunks

new_texts = [c["text"] for c in new_chunks]
new_embeddings = embedding_model.encode(new_texts)
new_ids = [f"chunk_{c['id']}" for c in new_chunks]
new_metadatas = [prepare_metadata(c["metadata"]) for c in new_chunks]

collection.add(
    ids=new_ids,
    documents=new_texts,
    embeddings=new_embeddings.tolist(),
    metadatas=new_metadatas
)

# Re-initialize BM25 with new chunks
tokenized_corpus = [t.lower().split() for t in new_texts]
bm25 = BM25Okapi(tokenized_corpus)

# Update global variables for search
chunks = new_chunks
metadatas = new_metadatas
print("System updated with table-row awareness!")

System updated with table-row awareness!


In [90]:
# # Test search again for the specific student
# query = "Deshmukh Pruthviraj Vijaysinh internship"
# hybrid_results = hybrid_search(query, top_k=5)

# final_results = rerank_results(query, hybrid_results, top_k=3)

# Run RAG
result = ask_document(
    query="When was UltraTech Building Products workshop organized?",
    collection=collection,
    chunk_map={i: {"text": chunks[i]["text"], "metadata": metadatas[i]} for i in range(len(chunks))},
    related_map={c["id"]: c for c in chunks},
    reranked_results=final_results
)

print("Answer:", result["answer"])

Answer: The information about UltraTech Building Products workshop is not available in the provided document.


In [96]:
def keyword_search(query, all_data, max_results=20):

    keywords = extract_query_terms(query)

    candidates = []

    for document, metadata in zip(
        all_data["documents"],
        all_data["metadatas"]
    ):

        text_lower = document.lower()

        matched_terms = [
            term
            for term in keywords
            if term in text_lower
        ]

        if matched_terms:

            score = len(matched_terms) / max(
                len(keywords), 1
            )

            candidates.append({
                "text": document,
                "metadata": metadata,
                "keyword_score": score,
                "matched_terms": matched_terms
            })

    candidates.sort(
        key=lambda x: x["keyword_score"],
        reverse=True
    )

    return candidates[:max_results]

In [97]:
def retrieve_candidates(query):

    semantic_results = semantic_search(query)

    keyword_results = keyword_search(
        query,
        all_data
    )

    candidates = merge_results(
        semantic_results,
        keyword_results
    )

    return candidates

In [98]:
def expand_element(element):

    element_type = element["metadata"].get(
        "element_type"
    )

    if element_type == "table_row":
        return expand_table(element)

    elif element_type == "image":
        return expand_image(element)

    elif element_type == "caption":
        return expand_caption(element)

    elif element_type == "section_header":
        return expand_section(element)

    else:
        return expand_text(element)

In [100]:
# Ask any question about the uploaded PDF

query = input("In which year “UltraTech Building Products” workshop was held")

answer = smart_ask(query)

print("\nAnswer:")
print(answer)

In which year “UltraTech Building Products” workshop was heldVishvas More research project under the guidance of??
Standard search failed. Retrying with high-recall keyword search...

Answer:
The information is not available in the provided document.
